# Modèles génératifs

**Objectifs.**
- Autoencodeur : apprendre une représentation compacte (espace latent) en reconstruisant l'entrée, et observer les limites de cette représentation pour la génération.
- *Conditional Flow Matching* (CFM) : un modèle génératif qui apprend un champ de vitesses transportant du bruit vers les données, entraîné par simple régression — et échantillonné en intégrant une équation différentielle.

Partie 1 sur MNIST (images), Partie 2 sur un jeu de données synthétique 2D (un mélange de 4 gaussiennes), pour visualiser directement ce que fait le modèle à l'échelle de la distribution de données.

In [ ]:
# !pip install -q torch torchvision matplotlib

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

import torchvision
from torchvision import transforms

import matplotlib.pyplot as plt

from training_toolbox import Trainer

torch.manual_seed(0)

## Partie 1 — Autoencodeur sur MNIST

Un autoencodeur apprend à reconstruire son entrée après être passé par une représentation
intermédiaire de dimension réduite (le **code**, ou représentation dans l'**espace
latent**). S'il arrive à reconstruire correctement l'image à partir d'un code beaucoup plus
petit que l'image elle-même, c'est que ce code capture l'essentiel de l'information utile.

**Question 1.1.** Implémentez la classe `AutoencoderDataset` ci-dessous. Quelle est la variable cible pour l'entraînement d'un autoencodeur ?

In [ ]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Lambda(torch.flatten)])

mnist_train = torchvision.datasets.MNIST(
    root="./data", train=True, download=True, transform=transform
)
mnist_test = torchvision.datasets.MNIST(
    root="./data", train=False, download=True, transform=transform
)


class AutoencoderDataset(Dataset):
    def __init__(self, base_dataset):
        self.base_dataset = base_dataset

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        img, _label = self.base_dataset[idx]
        # TODO : quelle est la variable cible pour l'entraînement d'un autoencodeur ?
        pass


ae_train_loader = DataLoader(AutoencoderDataset(mnist_train), batch_size=128, shuffle=True)
ae_test_loader = DataLoader(AutoencoderDataset(mnist_test), batch_size=256)

**Question 1.2.** Complétez `Autoencoder` : un encodeur qui réduit l'image (784 pixels
après aplatissement) à un code de dimension `latent_dim` via une couche cachée, et un
décodeur qui fait le chemin inverse. Entraînez-le pendant 8 epochs, et tracez les courbes de perte.

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, latent_dim=16, hidden_size=256):
        super().__init__()
        # TODO

    def forward(self, x):
        # TODO
        pass


autoencoder = Autoencoder(latent_dim=16)
optimizer = torch.optim.Adam(autoencoder.parameters(), lr=1e-3)
trainer = Trainer(autoencoder, optimizer, nn.MSELoss())
history = trainer.fit(ae_train_loader, ae_test_loader, epochs=8)
trainer.plot()

**Question 1.3.** Visualisez les reconstructions sur le jeu de test, et comparez-les aux images originales. Que constatez-vous ?

In [ ]:
def plot_reconstruction(model, images, n=6):
    model.eval()
    device = next(model.parameters()).device
    images = images.to(device)
    with torch.no_grad():
        reconstructions = model(images[:n])
    images = images.cpu()
    reconstructions = reconstructions.cpu()
    fig, axes = plt.subplots(2, n, figsize=(2 * n, 4))
    for i in range(n):
        axes[0, i].imshow(images[i].view(28, 28), cmap="gray")
        axes[0, i].axis("off")
        axes[1, i].imshow(reconstructions[i].view(28, 28), cmap="gray")
        axes[1, i].axis("off")
    plt.tight_layout()
    plt.show()


x_test_batch, _ = next(iter(ae_test_loader))
plot_reconstruction(autoencoder, x_test_batch)

## Partie 2 — Conditional Flow Matching sur un mélange de gaussiennes

Soit le jeu de données synthétique 2D suivant : un mélange de 4 gaussiennes.

**Question 2.1.** Complétez l'implémentation de la classe `FlowDataset` pour générer des échantillons de ce mélange de gaussiennes. On suppose ici que le modèle à entrainer prend en entrée un vecteur de dimension 3 : les 2 coordonnées de $x_t$ concaténées au temps $t$. Quelle est la variable cible ?

In [ ]:
n_samples = 3000
centers = torch.tensor([[4.0, 0.0], [0.0, 4.0], [-4.0, 0.0], [0.0, -4.0]])
std = 0.4

component = torch.randint(0, len(centers), (n_samples,))
data = centers[component] + std * torch.randn(n_samples, 2)

perm = torch.randperm(len(data))
n_val = 300
val_x = data[perm[:n_val]]
train_x = data[perm[n_val:]]

plt.figure(figsize=(4, 4))
plt.scatter(train_x[:, 0], train_x[:, 1], s=4, alpha=0.4)
plt.axis("equal")
plt.title("Données : mélange de 4 gaussiennes")
plt.show()


class FlowDataset(Dataset):
    def __init__(self, x1):
        self.x1 = x1

    def __len__(self):
        return len(self.x1)

    def __getitem__(self, idx):
        x1 = self.x1[idx]
        x0 = torch.randn(2)
        t = torch.rand(1)
        # TODO
        pass


cfm_train_loader = DataLoader(FlowDataset(train_x), batch_size=256, shuffle=True)
cfm_val_loader = DataLoader(FlowDataset(val_x), batch_size=256)

**Question 2.2.** Définissez `VelocityField` : un MLP qui prend en entrée un vecteur de
dimension 3 (les 2 coordonnées de $x_t$ concaténées au temps $t$). Ce sera votre modèle $u_\theta$ 
selon les notations du cours.
Deux couches cachées suffisent. Entraînez-le pendant 100 epochs, et tracez les courbes de perte.

In [ ]:
class VelocityField(nn.Module):
    def __init__(self, hidden_size=128):
        super().__init__()
        # TODO

    def forward(self, x):
        # TODO
        pass


velocity_model = VelocityField()
optimizer = torch.optim.Adam(velocity_model.parameters(), lr=1e-3)
cfm_trainer = Trainer(velocity_model, optimizer, nn.MSELoss())
cfm_history = cfm_trainer.fit(cfm_train_loader, cfm_val_loader, epochs=100, verbose=False)
cfm_trainer.plot()

**Question 2.3.** En utilisant la fonction fournie ci-dessous, générez 1000 points et
affichez-les sur le même graphique que les données réelles.

In [ ]:
@torch.no_grad()
def generate_samples(model, n_samples=1000, n_steps=200):
    model.eval()
    device = next(model.parameters()).device
    x = torch.randn(n_samples, 2, device=device)
    ts = torch.linspace(0, 1, n_steps + 1, device=device)
    dt = (ts[1] - ts[0]).item()
    for t in ts[:-1]:
        t_batch = torch.full((n_samples, 1), t.item(), device=device)
        v = model(torch.cat([x, t_batch], dim=1))
        x = x + dt * v
    return x.cpu()


# TODO

**Questions.**
- Contrairement à l'autoencodeur de la Partie 1, on peut ici partir d'un bruit purement
  aléatoire et obtenir des échantillons plausibles. Qu'est-ce que l'entraînement de CFM
  contraint, que l'entraînement de l'autoencodeur ne contraignait pas ?
- Que se passe-t-il si vous réduisez fortement `n_steps` (par exemple à 5) lors de la
  génération ? Pourquoi ?

_Votre réponse ici._

## Partie 3 — Observer la progression de l'entraînement

Plutôt que de ne regarder que le résultat final, sauvegardons le modèle à intervalles
réguliers pendant l'entraînement pour observer comment la qualité de génération évolue.
Contrairement au `ModelCheckpoint` que vous avez déjà utilisé, on veut ici plusieurs instantanés **successifs**, quel
que soit leur niveau de performance à cet instant, vous utiliserez pour cela la classe `PeriodicCheckpoint` fournie dans `training_toolbox.py
`. Vous pourrez ensuite générer des échantillons à partir de chacun de ces modèles sauvegardés, et les visualiser côte à côte pour observer la progression de l'entraînement.

In [ ]:
from training_toolbox import PeriodicCheckpoint
